# TradeSense · Component 03 · notebook 58: fixing the "too often cannot tell" result of notebook 49 (protocol C3 v1; CPU, minutes)

**Question.** Notebook 49 reported H-C4 as FAILED: only 44.6% of genuine minutes got the set {normal}. This notebook tests, under a new pre-registered protocol, whether building the set from what the detectors AGREE on (percentile of the fused, XGBoost and Transformer scores) makes the sets less undecided without losing coverage. Notebook 56 showed that smoothing the fused score alone does not. The results of notebooks 49 and 56 are not edited.

**Order of work.** (1) protocol hash, (2) CAL-3 data, (3) score chosen on CAL-3 only and tables written, (4) unit tests, (5) only then the saved TEST scores are read, (6) results. Nothing is retrained, no threshold changes.

**Protocol text hashed by this notebook:**

```
# Protocol C3 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 58 hashes this exact text) - conformal sets from model agreement (novelty 3, second follow-up to hypothesis H-C4 of protocol C1 v3)
Written BEFORE any number of this analysis exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. Everything below is an ADDITIONAL output on saved scores. CPU, no training. Protocol C1 v3 / notebook 49 (H-C4 FAILED) and protocol C2 v1 / notebook 56 (H-C4b and H-C5b FAILED) stay as reported and are not edited. The hypothesis thresholds below are the SAME as in C2 v1; they are not lowered to make the test easier.

## Why C2 failed (diagnosis from notebook 56)
A smoother or a percentile transform of the SAME fused score cannot separate a weak event minute from an ordinary minute, because both have a similar fused score; all four candidates of notebook 56 gave the same mix. The missing information is what the detectors AGREE on. A genuine spike usually lifts one detector (typically the one that reacts to a single feature) while a real event pattern lifts the supervised detectors together. This is the same idea as the no-conflict check already used by the reliability rules.

## Part A - candidate scores (declared now; chosen on CAL-3 only)
For each of the series F (fused), X (XGBoost v2), T (Transformer v2) let u be the percentile of the value within the trailing 7-day genuine scores of the SAME series strictly before t (the drift-free transform of C2 v1, quantile grid 0.01, 0.05, 0.10, 0.20, ..., 0.90, 0.95, 0.99, 0.999; below / above the grid -> 0.005 / 0.9995). AE-ctx is excluded as in the no-conflict check (it is already inside F). Candidates (all causal; same function for event and genuine series):
- u: u_F alone (the result of C2 v1, kept as reference).
- m3: mean of u_F, u_X, u_T.
- min2: min of u_X and u_T (both supervised detectors must be high).
- m3s5: mean of m3 over the last 5 bars.
Sets: Mondrian split conformal exactly as in C1 v3 / C2 v1 but in the space of the candidate score; normal p = (1 + #{calibration genuine score >= s_t}) / (n + 1); abnormal p = (1 + #{calibration abnormal score <= s_t}) / (n + 1); Gamma = {y : p_y > eps}, eps = 0.10 on both sides.
Selection (CAL-3 only, TEST not loaded): CAL-3 bars 1,440..44,639 split chronologically, tables fitted on the first 60%, evaluated on the last 40%, per symbol, averaged over BTC and ETH. Rule: among candidates with both held-out coverages (normal, abnormal) >= 0.85, choose the highest share of genuine bars with Gamma = {normal}; ties go to the simpler candidate in the order u, m3, min2, m3s5; if none qualifies the choice is u. Tables of the chosen candidate are refitted on the whole CAL-3 and written to declared_conformal_v3.json / conformal_tables_v3.npz BEFORE any TEST bar is read, and never changed.

## Part B - hypotheses on saved TEST-B scores (Sep and Oct 2024 genuine bars; TEST-3 event bars)
Baseline B0 = variant R of C1 v3 recomputed in the same notebook on the same bars (valid bars for both methods).
- H-C1c normal coverage >= 0.88 in all 4 cells (symbol x month).
- H-C2c share of event bars with abnormal in Gamma >= 0.85 (mean of BTC and ETH).
- H-C3c LR(abnormal-only set) >= 3 (pooled).
- H-C4c share of genuine bars with Gamma = {normal} >= 0.55 AND the paired improvement over B0 has a day-block bootstrap 95% lower bound > 0 (units = symbol x day, 2,000 draws, seed 0).
- H-C5c share of genuine bars with Gamma = cannot tell <= 0.35.
- Descriptive (no decision): the trade-off curve for eps on the abnormal side in {0.05, 0.10, 0.20, 0.30, 0.40}; the same table for the other three candidates; set mix on Sep vs Oct; share of ALERT bars with Gamma = {abnormal}.
- Stated limits: autocorrelated bars and drift break exchangeability, coverage is approximate and tested out of sample; labels are synthetic; sets describe the SCORE, not whether manipulation happened. If a hypothesis fails it is reported as failed and nothing is retuned; if it fails, the structural limit (weak event minutes) is the documented answer and the engine keeps variant R.
- If the chosen candidate holds H-C1c..H-C5c, the engine computes it causally from the per-model score histories it already keeps (additive output, no frozen part changes).

```


In [1]:
import os, sys, json, time, hashlib, warnings, glob
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_C3 = r"""# Protocol C3 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 58 hashes this exact text) - conformal sets from model agreement (novelty 3, second follow-up to hypothesis H-C4 of protocol C1 v3)
Written BEFORE any number of this analysis exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. Everything below is an ADDITIONAL output on saved scores. CPU, no training. Protocol C1 v3 / notebook 49 (H-C4 FAILED) and protocol C2 v1 / notebook 56 (H-C4b and H-C5b FAILED) stay as reported and are not edited. The hypothesis thresholds below are the SAME as in C2 v1; they are not lowered to make the test easier.

## Why C2 failed (diagnosis from notebook 56)
A smoother or a percentile transform of the SAME fused score cannot separate a weak event minute from an ordinary minute, because both have a similar fused score; all four candidates of notebook 56 gave the same mix. The missing information is what the detectors AGREE on. A genuine spike usually lifts one detector (typically the one that reacts to a single feature) while a real event pattern lifts the supervised detectors together. This is the same idea as the no-conflict check already used by the reliability rules.

## Part A - candidate scores (declared now; chosen on CAL-3 only)
For each of the series F (fused), X (XGBoost v2), T (Transformer v2) let u be the percentile of the value within the trailing 7-day genuine scores of the SAME series strictly before t (the drift-free transform of C2 v1, quantile grid 0.01, 0.05, 0.10, 0.20, ..., 0.90, 0.95, 0.99, 0.999; below / above the grid -> 0.005 / 0.9995). AE-ctx is excluded as in the no-conflict check (it is already inside F). Candidates (all causal; same function for event and genuine series):
- u: u_F alone (the result of C2 v1, kept as reference).
- m3: mean of u_F, u_X, u_T.
- min2: min of u_X and u_T (both supervised detectors must be high).
- m3s5: mean of m3 over the last 5 bars.
Sets: Mondrian split conformal exactly as in C1 v3 / C2 v1 but in the space of the candidate score; normal p = (1 + #{calibration genuine score >= s_t}) / (n + 1); abnormal p = (1 + #{calibration abnormal score <= s_t}) / (n + 1); Gamma = {y : p_y > eps}, eps = 0.10 on both sides.
Selection (CAL-3 only, TEST not loaded): CAL-3 bars 1,440..44,639 split chronologically, tables fitted on the first 60%, evaluated on the last 40%, per symbol, averaged over BTC and ETH. Rule: among candidates with both held-out coverages (normal, abnormal) >= 0.85, choose the highest share of genuine bars with Gamma = {normal}; ties go to the simpler candidate in the order u, m3, min2, m3s5; if none qualifies the choice is u. Tables of the chosen candidate are refitted on the whole CAL-3 and written to declared_conformal_v3.json / conformal_tables_v3.npz BEFORE any TEST bar is read, and never changed.

## Part B - hypotheses on saved TEST-B scores (Sep and Oct 2024 genuine bars; TEST-3 event bars)
Baseline B0 = variant R of C1 v3 recomputed in the same notebook on the same bars (valid bars for both methods).
- H-C1c normal coverage >= 0.88 in all 4 cells (symbol x month).
- H-C2c share of event bars with abnormal in Gamma >= 0.85 (mean of BTC and ETH).
- H-C3c LR(abnormal-only set) >= 3 (pooled).
- H-C4c share of genuine bars with Gamma = {normal} >= 0.55 AND the paired improvement over B0 has a day-block bootstrap 95% lower bound > 0 (units = symbol x day, 2,000 draws, seed 0).
- H-C5c share of genuine bars with Gamma = cannot tell <= 0.35.
- Descriptive (no decision): the trade-off curve for eps on the abnormal side in {0.05, 0.10, 0.20, 0.30, 0.40}; the same table for the other three candidates; set mix on Sep vs Oct; share of ALERT bars with Gamma = {abnormal}.
- Stated limits: autocorrelated bars and drift break exchangeability, coverage is approximate and tested out of sample; labels are synthetic; sets describe the SCORE, not whether manipulation happened. If a hypothesis fails it is reported as failed and nothing is retuned; if it fails, the structural limit (weak event minutes) is the documented answer and the engine keeps variant R.
- If the chosen candidate holds H-C1c..H-C5c, the engine computes it causally from the per-model score histories it already keeps (additive output, no frozen part changes).
"""
PHC = hashlib.sha256(PROTO_C3.encode()).hexdigest(); pp = OUT/'protocol_c3_v1.md'
if pp.exists(): assert pp.read_text() == PROTO_C3, 'protocol C3 v1 exists and differs from this notebook'
else: pp.write_text(PROTO_C3)
(OUT/'protocol_c3_v1.sha256').write_text(PHC + '  protocol_c3_v1.md\n'); assert (OUT/'protocol_c1_v3.sha256').exists() and (OUT/'c1_results.json').exists(), 'run notebook 49 first'
for f in ['test_opened_v1.json', 'test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'b2_lib.py', 'c03_infer.py']:
    assert (OUT/f).exists(), f'{f} missing'
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
for s in SYMBOLS: assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'
assert sha_file(OUT/'c03_infer.py') == T32['library_sha16'] and sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'library files differ from the ones that opened TEST'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
TEST_LOADED = False
say('protocol C3 v1 sha256 ' + PHC[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + T32['library_sha16'] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol C3 v1 sha256 a0dbd8ca67913439; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9


## 1 · Definitions (from notebook 49, plus the new transform)

In [2]:
EPS = 0.10; EPS_ALT = [0.05, 0.10, 0.20]; WARM = B.WARM
AOLEN = 31 * 1440; CAL_LO = 1440; AOFULL = 132480; TEST_LO = AOLEN; SEP_HI = TEST_LO + 30 * 1440
def thr_roll(g, q): return c.rolling_threshold(g, q)
# ---- conformal p-values (class-conditional, split)
def p_norm_S(x, cal_norm):   # (1 + #{calibration normal bars with score >= x}) / (n + 1)
    return (1 + (len(cal_norm) - np.searchsorted(cal_norm, x, side='left'))) / (len(cal_norm) + 1)
def p_abn_S(x, cal_abn):     # (1 + #{calibration abnormal bars with score <= x}) / (n + 1)
    return (1 + np.searchsorted(cal_abn, x, side='right')) / (len(cal_abn) + 1)
def persistence(alert, k=5, need=3):
    cs = np.concatenate([[0], np.cumsum(alert.astype(np.int64))]); i = np.arange(len(alert)); return (cs[i + 1] - cs[np.maximum(i - k + 1, 0)]) >= need
def dq_ok_from(df, hist=1440, last=15):
    X = df[B.FEATS].to_numpy('float64'); bad = ~np.isfinite(X).all(axis=1) | (df['volume'].to_numpy('float64') <= 0) | (df['trades_count'].to_numpy('float64') <= 0)
    cs = np.concatenate([[0], np.cumsum(bad.astype(np.int64))]); i = np.arange(len(df)); ok = (cs[i + 1] - cs[np.maximum(i - last + 1, 0)]) == 0; ok[:hist] = False; return ok
def levels(alert, rare, pers, noconf, dq):
    """3 Strong (gate + all 3 checks), 2 Moderate (gate + exactly 2), 1 Limited (other Alert bars), 0 not an Alert bar"""
    n = rare.astype(int) + pers.astype(int) + noconf.astype(int); strong = alert & dq & (n == 3); mod = alert & dq & (n == 2); lim = alert & ~strong & ~mod
    return np.where(strong, 3, np.where(mod, 2, np.where(lim, 1, 0)))
class Src:
    """one symbol, one evaluation period: fused and model scores of the injected and the genuine series + labels + data-quality flag"""
    def __init__(self, name, s, F_inj, F_gen, M_inj, M_gen, lab, inseg, dq, lo, hi):
        self.name, self.s, self.F_inj, self.F_gen, self.M_inj, self.M_gen, self.lab, self.inseg, self.dq, self.lo, self.hi = name, s, F_inj, F_gen, M_inj, M_gen, lab, inseg, dq, lo, hi
        self.n = len(F_inj); assert len(F_gen) == self.n == len(lab) == len(dq) and all(len(M_inj[m]) == self.n == len(M_gen[m]) for m in M_inj)
        self.idx = np.arange(self.n); self.rng = (self.idx >= lo) & (self.idx < hi); self.pos = (lab > 0) & inseg & (self.idx >= WARM) & self.rng
    def copy_with(self, t0, mult=3.0):
        f = lambda a: np.where(np.arange(len(a)) >= t0, a * mult + 1.0, a)
        return Src(self.name, self.s, f(self.F_inj), f(self.F_gen), {m: f(v) for m, v in self.M_inj.items()}, {m: f(v) for m, v in self.M_gen.items()}, self.lab, self.inseg, self.dq, self.lo, self.hi)
def assess(src, tab, variant='R', eps=EPS):
    s = src.s; qF = QF[s]; thrA = thr_roll(src.F_gen, qF); out = {}
    for tag, F in [('inj', src.F_inj), ('gen', src.F_gen)]:
        if variant == 'R':
            tn = thr_roll(src.F_gen, 1 - eps); hist = np.isfinite(tn); nin = (F <= tn) & hist
        else:
            hist = np.ones(len(F), bool); nin = p_norm_S(np.nan_to_num(F, nan=-1e9), tab[s]['norm']) > eps
        ain = p_abn_S(np.nan_to_num(F, nan=-1e9), tab[s]['abn']) > eps
        gam = nin.astype(int) + 2 * ain.astype(int); alert = (F > thrA)
        M = src.M_inj if tag == 'inj' else src.M_gen
        above = {m: M[m] > thr_roll(src.M_gen[m], QM[m][s]) for m in M}; low = {m: M[m] <= thr_roll(src.M_gen[m], 0.90) for m in M}
        x, t = 'XGBoost v2', 'Transformer v2'; conf = (above[x] & low[t]) | (above[t] & low[x]); pers = persistence(alert)
        p001 = F > thr_roll(src.F_gen, 0.999); lv = levels(alert, p001, pers, ~conf, src.dq); agree = sum(above[m].astype(int) for m in M)
        out[tag] = {'gam': gam, 'alert': alert, 'pers': pers, 'conf': conf, 'lv': lv, 'hist': hist, 'agree': agree, 'p001': p001}
    return out
# ---------------- new in C2: drift-free percentile transform and its causal smoothings
QGRID = np.array([0.01, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99, 0.999]); CANDS = ['u', 'm3', 'min2', 'm3s5']
def useries(src):
    """percentiles u of F, XGBoost, Transformer for the event and the genuine series (each within the trailing 7-day genuine scores of its own series)"""
    if not hasattr(src, '_us'):
        X, T_ = MODELS[1], MODELS[2]; us = {}
        for key, Fi, Fg in [('F', src.F_inj, src.F_gen), ('X', src.M_inj[X], src.M_gen[X]), ('T', src.M_inj[T_], src.M_gen[T_])]:
            qg = np.column_stack([thr_roll(Fg, q) for q in QGRID]); us[key] = (u_of(Fi, qg), u_of(Fg, qg))
        src._us = us
    return src._us
def u_of(F, qg):
    """percentile of F[t] within the trailing 7-day genuine scores before t (piecewise-linear over QGRID); NaN without history"""
    n = len(F); u = np.full(n, np.nan); ok = np.isfinite(F) & np.isfinite(qg).all(axis=1)
    Fo, Q = np.asarray(F)[ok], qg[ok]; G = len(QGRID); k = (Fo[:, None] > Q).sum(axis=1); res = np.empty(len(Fo))
    lo, hi = k == 0, k == G; mid = ~lo & ~hi; res[lo] = QGRID[0] / 2; res[hi] = (1 + QGRID[-1]) / 2
    ii = k[mid]; r = np.arange(mid.sum()); q0 = Q[mid][r, ii - 1]; q1 = Q[mid][r, ii]; w = np.where(q1 > q0, (Fo[mid] - q0) / np.where(q1 > q0, q1 - q0, 1.0), 0.0)
    res[mid] = QGRID[ii - 1] + w * (QGRID[ii] - QGRID[ii - 1]); u[ok] = res; return u
def TT(src, name):
    key = '_T_' + name
    if not hasattr(src, key):
        us = useries(src); res = []
        for j in (0, 1):
            f, x, t = us['F'][j], us['X'][j], us['T'][j]
            if name == 'u': v = f
            elif name == 'min2': v = np.minimum(x, t)
            else:
                v = (f + x + t) / 3
                if name == 'm3s5': v = pd.Series(v).rolling(5, min_periods=5).mean().to_numpy()
            res.append(v)
        setattr(src, key, tuple(res))
    return getattr(src, key)
def fit_tab(src, name, lo, hi):
    Ti, Tg = TT(src, name); i = src.idx; pos = src.pos & (i >= lo) & (i < hi) & np.isfinite(Ti); gen = src.rng & (i >= lo) & (i < hi) & (i >= WARM) & np.isfinite(Tg)
    return {'abn': np.sort(Ti[pos]), 'norm': np.sort(Tg[gen])}
def gamma_U(src, name, tab, eps_n=EPS, eps_a=EPS):
    """Gamma codes (1 normal only, 2 abnormal only, 3 cannot tell, 0 neither) for the event series and the genuine series, plus the validity masks"""
    out = {}
    for tag, T in zip(['inj', 'gen'], TT(src, name)):
        ok = np.isfinite(T); x = np.nan_to_num(T, nan=-1.0)
        out[tag] = {'gam': (p_norm_S(x, tab['norm']) > eps_n).astype(int) + 2 * (p_abn_S(x, tab['abn']) > eps_a).astype(int), 'ok': ok}
    return out


## 2 · CAL-3 data

In [3]:
DF = json.load(open(OUT/'declared_final_set_v1.json')); AE2 = json.load(open(OUT/'declared_ae_v2.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json'))
MODELS = c.MODEL_NAMES; FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {'AE-ctx fmax': 'ae_ctx_agg', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}
QM = {MODELS[0]: AE2['q_T1_by_budget']['2'], MODELS[1]: DF['models'][MODELS[1]]['q_T1_by_budget']['2'], MODELS[2]: DF['models'][MODELS[2]]['q_T1_by_budget']['2']}; QF = FUSD['q_T1_by_budget']['2']
TAGM = dict(zip(MODELS, ['AE', 'XGB', 'TR']))
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:WARM] = np.nan; return a
def read_par(s, name, n):
    p = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet'); p = p[p.replica == 0].reset_index(drop=True).iloc[:n]; assert len(p) == n, (s, name, len(p)); return p
CALSRC = {}; CALZ = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); assert not any('TEST' in k for k in z.files); CALZ[s] = {k: z[k] for k in z.files}
    Mi, Mg = {}, {}
    for m in MODELS:
        zz = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'); assert not any('TEST' in k for k in zz.files)
        gi = lambda k: (lambda x: x[:, 0] if x.ndim == 2 else x)(zz[k])
        Mi[m] = nanw(gi('CAL_r0')[:AOLEN]); Mg[m] = nanw(gi('GEN_AO'))
    par = read_par(s, 'CAL', AOLEN); dq = dq_ok_from(par)
    CALSRC[s] = Src('CAL', s, nanw(CALZ[s]['CAL_r0'][:AOLEN]), nanw(CALZ[s]['GEN_AO']), Mi, Mg, par.label.to_numpy().astype(int), par.in_seg.to_numpy().astype(bool), dq, CAL_LO, AOLEN)
say('CAL-3 loaded: positive (event) bars per symbol ' + str({s: int(CALSRC[s].pos.sum()) for s in SYMBOLS}) + '; share of bars failing the data-quality check ' + str({s: round(float((~CALSRC[s].dq[CAL_LO:]).mean()), 4) for s in SYMBOLS}))

CAL-3 loaded: positive (event) bars per symbol {'BTCUSDT': 5546, 'ETHUSDT': 5658}; share of bars failing the data-quality check {'BTCUSDT': 0.0, 'ETHUSDT': 0.0}


## 3 · Score chosen on CAL-3 only, tables written before TEST

In [4]:

# ---- the declared tables of notebook 49 (baseline B0)
_z1 = np.load(OUT/'conformal_tables_v1.npz'); TAB = {s: {'abn': _z1[f'{s}_abn'], 'norm': _z1[f'{s}_norm']} for s in SYMBOLS}
# ---- candidate selection on CAL-3 only: first 60% fit, last 40% held-out
SPLIT = CAL_LO + int(0.6 * (AOLEN - CAL_LO)); SEL = []
for nm in CANDS:
    r_ = []
    for s in SYMBOLS:
        S_ = CALSRC[s]; tab = fit_tab(S_, nm, CAL_LO, SPLIT); G_ = gamma_U(S_, nm, tab); ev = (S_.idx >= SPLIT) & (S_.idx < AOLEN)
        gm = ev & S_.rng & (S_.idx >= WARM) & G_['gen']['ok']; pm = ev & S_.pos & G_['inj']['ok']; gg = G_['gen']['gam'][gm]; gi = G_['inj']['gam'][pm]
        r_.append([float(((gg & 1) > 0).mean()), float(((gi & 2) > 0).mean()), float((gg == 1).mean()), float((gg == 3).mean())])
    m = np.mean(r_, axis=0); SEL.append({'candidate': nm, 'held-out normal coverage': m[0], 'held-out abnormal coverage': m[1], 'genuine share {normal}': m[2], 'genuine share cannot tell': m[3]})
SELT = pd.DataFrame(SEL); ok_ = SELT[(SELT['held-out normal coverage'] >= 0.85) & (SELT['held-out abnormal coverage'] >= 0.85)]
CHOSEN = CANDS[0] if len(ok_) == 0 else ok_.sort_values('genuine share {normal}', ascending=False, kind='stable').iloc[0].candidate
ties = ok_[ok_['genuine share {normal}'] >= ok_['genuine share {normal}'].max() - 1e-12].candidate.tolist() if len(ok_) else []
if len(ties) > 1: CHOSEN = [c_ for c_ in CANDS if c_ in ties][0]
say('--- candidate selection on CAL-3 (fit first 60%, held-out last 40%; TEST not loaded) ---'); say(SELT.round(3).to_string(index=False)); say(f'chosen score: {CHOSEN}' + ('' if len(ok_) else '  (no candidate reached both held-out coverages >= 0.85: default u)'))
# ---- tables of the chosen score on the whole CAL-3, written BEFORE any TEST bar is read
TAB2 = {s: fit_tab(CALSRC[s], CHOSEN, CAL_LO, AOLEN) for s in SYMBOLS}
np.savez_compressed(OUT/'conformal_tables_v3_new.npz', **{f'{s}_{k}': v for s in SYMBOLS for k, v in TAB2[s].items()}); tp2 = OUT/'conformal_tables_v3.npz'
if tp2.exists():
    old = np.load(tp2)
    for s in SYMBOLS:
        for k in ['abn', 'norm']: assert np.array_equal(old[f'{s}_{k}'], TAB2[s][k]), 'declared v2 tables exist and differ from the ones recomputed from CAL-3'
    (OUT/'conformal_tables_v3_new.npz').unlink()
else: (OUT/'conformal_tables_v3_new.npz').rename(tp2)
DECL2 = {'status': 'DECLARED before any TEST bar is read in notebook 58', 'protocol_c3_v1_sha256': PHC, 'transform': CHOSEN, 'quantile_grid': QGRID.tolist(), 'eps': EPS, 'selection_table': SELT.to_dict('records'), 'table_sizes': {s: {k: int(len(v)) for k, v in TAB2[s].items()} for s in SYMBOLS}, 'tables_file': 'conformal_tables_v3.npz', 'tables_sha16': sha_file(tp2),
         'rules': 'u = percentile of the fused score within the trailing 7-day genuine fused scores before t (piecewise-linear over the grid; below/above the grid -> 0.005 / 0.9995); T = chosen causal smoothing of u; normal in Gamma iff p_norm > eps; abnormal in Gamma iff p_abn > eps (tables of T from CAL-3)'}
dp2 = OUT/'declared_conformal_v3.json'
if dp2.exists(): assert json.loads(dp2.read_text())['tables_sha16'] == DECL2['tables_sha16'], 'declared v2 tables changed'
else: dp2.write_text(json.dumps(DECL2, indent=1))
assert not TEST_LOADED; say('conformal tables v3 written (before TEST): ' + str(DECL2['table_sizes']) + '; sha16 ' + DECL2['tables_sha16'])


--- candidate selection on CAL-3 (fit first 60%, held-out last 40%; TEST not loaded) ---
candidate  held-out normal coverage  held-out abnormal coverage  genuine share {normal}  genuine share cannot tell
        u                     0.866                       0.930                   0.340                      0.526
       m3                     0.862                       0.926                   0.355                      0.507
     min2                     0.861                       0.929                   0.319                      0.542
     m3s5                     0.867                       0.928                   0.318                      0.548
chosen score: m3
conformal tables v3 written (before TEST): {'BTCUSDT': {'abn': 5546, 'norm': 43170}, 'ETHUSDT': {'abn': 5658, 'norm': 43170}}; sha16 ce5eb46ab4bbfc16


## 4 · Unit tests

In [5]:

# T01 the percentile transform: uniform genuine data -> u close to uniform; a value above everything -> top of the grid
rg = np.random.default_rng(3); g_ = rg.normal(size=20000); qg_ = np.column_stack([thr_roll(g_, q) for q in QGRID]); u_ = u_of(g_, qg_); v_ = u_[~np.isnan(u_)]
assert abs(np.mean(v_ <= 0.5) - 0.5) < 0.03 and abs(np.mean(v_ <= 0.9) - 0.9) < 0.03 and np.isclose(u_of(np.array([1e9] * 3), np.tile(qg_[-1], (3, 1)))[0], (1 + QGRID[-1]) / 2)
say('PASS T01 percentile transform: on stationary data it is uniform within 0.03 and maps a value above the whole grid to 0.9995')
# T02 causality: changing every score from bar t0 on leaves earlier T values unchanged
S0 = CALSRC[SYMBOLS[0]]; S1 = S0.copy_with(30000)
for nm in CANDS:
    for a_, b_ in zip(TT(S0, nm), TT(S1, nm)): assert np.array_equal(a_[:30000], b_[:30000], equal_nan=True), nm
say('PASS T02 every candidate transform at bar t does not depend on later bars')
# T03 on exchangeable synthetic data both class coverages are 1 - eps (the same property as T02 of notebook 49, in T-space)
cn_ = np.sort(rg.uniform(size=3000)); ca_ = np.sort(rg.beta(5, 1.5, 3000)); tn2, ta2 = rg.uniform(size=40000), rg.beta(5, 1.5, 40000)
assert abs((p_norm_S(tn2, cn_) > 0.1).mean() - 0.9) < 0.015 and abs((p_abn_S(ta2, ca_) > 0.1).mean() - 0.9) < 0.015; say('PASS T03 coverage on exchangeable synthetic data equals 1 - eps within 0.015')
# T04 the selection used CAL-3 only (TEST not loaded yet)
assert not TEST_LOADED; say('PASS T04 selection and tables use CAL-3 only; no TEST bar has been read')


PASS T01 percentile transform: on stationary data it is uniform within 0.03 and maps a value above the whole grid to 0.9995
PASS T02 every candidate transform at bar t does not depend on later bars
PASS T03 coverage on exchangeable synthetic data equals 1 - eps within 0.015
PASS T04 selection and tables use CAL-3 only; no TEST bar has been read


## 5 · Saved TEST scores (read only now)

In [6]:
# ---- only now are the saved TEST scores read
TEST_LOADED = True
TSRC = {}; EVT = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'test_b_{s}.npz'); par = read_par(s, 'TEST', AOFULL)
    Mi = {m: nanw(z[TAGM[m] + '__r0']) for m in MODELS}; Mg = {m: nanw(z[TAGM[m] + '__GEN']) for m in MODELS}
    TSRC[s] = Src('TEST', s, nanw(z['FUS__r0']), nanw(z['FUS__GEN']), Mi, Mg, par.label.to_numpy().astype(int), par.in_seg.to_numpy().astype(bool), dq_ok_from(par), TEST_LO, AOFULL)
    EVT[s] = pd.read_csv(L3/f'l3v3_{s}_TEST_events.csv'); EVT[s] = EVT[s][EVT[s].replica == 0].reset_index(drop=True)
say('saved TEST scores loaded: event bars per symbol ' + str({s: int(TSRC[s].pos.sum()) for s in SYMBOLS}) + '; genuine TEST-B days ' + str((AOFULL - TEST_LO) // 1440) + '; bars failing data quality in TEST ' + str({s: int((~TSRC[s].dq[TEST_LO:]).sum()) for s in SYMBOLS}))

saved TEST scores loaded: event bars per symbol {'BTCUSDT': 8366, 'ETHUSDT': 8379}; genuine TEST-B days 61; bars failing data quality in TEST {'BTCUSDT': 0, 'ETHUSDT': 0}


## 6 · Results (protocol C3 v1)

In [7]:

RES = {}; AS = {}; AU = {}
for s in SYMBOLS: AS[s] = assess(TSRC[s], TAB, 'R'); AU[s] = gamma_U(TSRC[s], CHOSEN, TAB2[s])
MON = {'Sep': (TEST_LO, SEP_HI), 'Oct': (SEP_HI, AOFULL)}
def gmask(s, lo, hi):
    S_ = TSRC[s]; m = np.zeros(S_.n, bool); m[lo:hi] = True; return m & np.isfinite(S_.F_gen) & AS[s]['gen']['hist'] & AU[s]['gen']['ok'] & (S_.idx >= WARM)     # bars valid for BOTH methods
def pmask(s): S_ = TSRC[s]; return S_.pos & np.isfinite(S_.F_inj) & AS[s]['inj']['hist'] & AU[s]['inj']['ok']
# ---------- coverage per cell
rows = []
for s in SYMBOLS:
    for mon, (lo, hi) in MON.items():
        g = gmask(s, lo, hi); rows.append({'sym': s, 'month': mon, 'bars': int(g.sum()), 'B0 normal coverage': float(((AS[s]['gen']['gam'][g] & 1) > 0).mean()), 'C3 normal coverage': float(((AU[s]['gen']['gam'][g] & 1) > 0).mean()), 'target': 1 - EPS})
COV = pd.DataFrame(rows); COV.to_csv(OUT/'c3_coverage_table.csv', index=False); say('--- normal-class coverage on genuine TEST-B bars (B0 = variant R of C1 v3; C3 = chosen score ' + CHOSEN + ') ---'); say(COV.round(3).to_string(index=False))
HC1b = bool((COV['C3 normal coverage'] >= 0.88).all())
a_b0 = float(np.mean([((AS[s]['inj']['gam'][pmask(s)] & 2) > 0).mean() for s in SYMBOLS])); a_c2 = float(np.mean([((AU[s]['inj']['gam'][pmask(s)] & 2) > 0).mean() for s in SYMBOLS])); HC2b = bool(a_c2 >= 0.85)
# ---------- set mix, H-C3c, H-C4c, H-C5c
Gb, Gc, Eb, Ec, UNIT = [], [], [], [], []
for s in SYMBOLS:
    g = gmask(s, TEST_LO, AOFULL); Gb.append(AS[s]['gen']['gam'][g]); Gc.append(AU[s]['gen']['gam'][g]); p = pmask(s); Eb.append(AS[s]['inj']['gam'][p]); Ec.append(AU[s]['inj']['gam'][p]); day = (np.arange(TSRC[s].n) - TEST_LO) // 1440
    for d_ in range((AOFULL - TEST_LO) // 1440):
        m_ = g & (day == d_); UNIT.append([int(m_.sum()), int((AS[s]['gen']['gam'][m_] == 1).sum()), int((AU[s]['gen']['gam'][m_] == 1).sum())])
Gb, Gc, Eb, Ec, UNIT = np.concatenate(Gb), np.concatenate(Gc), np.concatenate(Eb), np.concatenate(Ec), np.array(UNIT)
names = {1: '{normal}', 3: 'cannot tell', 2: '{abnormal}', 0: '{} unlike both'}; mixb = {names[k]: float((Gb == k).mean()) for k in names}; mixc = {names[k]: float((Gc == k).mean()) for k in names}
say(''); say('set mix on genuine TEST-B bars (eps 0.10):'); say('  B0 : ' + ', '.join(f'{k} {v:.3f}' for k, v in mixb.items())); say('  C3 : ' + ', '.join(f'{k} {v:.3f}' for k, v in mixc.items()))
LR3 = float((Ec == 2).mean() / max((Gc == 2).mean(), 1e-12)); LR3b = float((Eb == 2).mean() / max((Gb == 2).mean(), 1e-12)); HC3b = bool(LR3 >= 3)
say(f'abnormal coverage on event bars: B0 {a_b0:.3f}, C3 {a_c2:.3f}; LR of the abnormal-only set: B0 {LR3b:.2f}, C3 {LR3:.2f}')
rg = np.random.default_rng(0); dif = []
for _ in range(NBOOT):
    pk = rg.integers(0, len(UNIT), len(UNIT)); t_ = UNIT[pk].sum(axis=0); dif.append(t_[2] / t_[0] - t_[1] / t_[0])
ci = np.percentile(dif, [2.5, 97.5]); gain = mixc['{normal}'] - mixb['{normal}']; HC4b = bool(mixc['{normal}'] >= 0.55 and ci[0] > 0); HC5b = bool(mixc['cannot tell'] <= 0.35)
# ---------- descriptive: trade-off curve, original target, other candidates, Sep vs Oct, Alert bars
curve = []
for ea in [0.05, 0.10, 0.20, 0.30, 0.40]:
    gg, ee = [], []
    for s in SYMBOLS:
        Gx = gamma_U(TSRC[s], CHOSEN, TAB2[s], EPS, ea); gg.append(Gx['gen']['gam'][gmask(s, TEST_LO, AOFULL)]); ee.append(Gx['inj']['gam'][pmask(s)])
    gg, ee = np.concatenate(gg), np.concatenate(ee); curve.append({'eps abnormal side': ea, 'abnormal coverage': float(((ee & 2) > 0).mean()), 'genuine {normal}': float((gg == 1).mean()), 'genuine cannot tell': float((gg == 3).mean()), 'LR abnormal-only': float((ee == 2).mean() / max((gg == 2).mean(), 1e-12))})
CV = pd.DataFrame(curve); CV.to_csv(OUT/'c3_tradeoff_curve.csv', index=False); say(''); say('--- trade-off curve (chosen score, normal side eps 0.10; descriptive) ---'); say(CV.round(3).to_string(index=False))
oth = []
for nm in CANDS:
    tabn = {s: fit_tab(CALSRC[s], nm, CAL_LO, AOLEN) for s in SYMBOLS}; gg, ee = [], []
    for s in SYMBOLS:
        Gx = gamma_U(TSRC[s], nm, tabn[s]); ok_g = gmask(s, TEST_LO, AOFULL) & Gx['gen']['ok']; ok_p = pmask(s) & Gx['inj']['ok']; gg.append(Gx['gen']['gam'][ok_g]); ee.append(Gx['inj']['gam'][ok_p])
    gg, ee = np.concatenate(gg), np.concatenate(ee); oth.append({'transform': nm, 'normal coverage': float(((gg & 1) > 0).mean()), 'abnormal coverage': float(((ee & 2) > 0).mean()), 'genuine {normal}': float((gg == 1).mean()), 'LR abnormal-only': float((ee == 2).mean() / max((gg == 2).mean(), 1e-12))})
say(''); say('--- all four transforms on TEST-B (descriptive; only the one chosen on CAL-3 is tested) ---'); say(pd.DataFrame(oth).round(3).to_string(index=False))
mo = []
for mon, (lo, hi) in MON.items():
    gg = np.concatenate([AU[s]['gen']['gam'][gmask(s, lo, hi)] for s in SYMBOLS]); mo.append({'month': mon, **{names[k]: float((gg == k).mean()) for k in names}})
say(''); say('--- set mix by month (C3) ---'); say(pd.DataFrame(mo).round(3).to_string(index=False))
al = np.concatenate([AU[s]['inj']['gam'][AS[s]['inj']['alert'] & pmask(s)] for s in SYMBOLS]); say(f'share of event Alert bars with Gamma = abnormal-only (C3): {float((al == 2).mean()):.3f}; original H-C4 target (share {{normal}} >= 0.80, descriptive): {mixc["{normal}"]:.3f}')
say(''); say('=== pre-registered hypotheses (protocol C3 v1; new analysis on saved TEST scores) ===')
say(f'H-C1c normal coverage >= 0.88 in all 4 cells: min {COV["C3 normal coverage"].min():.3f} -> {"HOLDS" if HC1b else "FAILS"}')
say(f'H-C2c abnormal coverage on event bars >= 0.85: {a_c2:.3f} -> {"HOLDS" if HC2b else "FAILS"}')
say(f'H-C3c LR(abnormal-only set) >= 3: {LR3:.2f} -> {"HOLDS" if HC3b else "FAILS"}')
say(f'H-C4c genuine {{normal}} >= 0.55 and paired gain over B0 with lower bound > 0: {mixc["{normal}"]:.3f} (B0 {mixb["{normal}"]:.3f}), gain {gain:+.3f} ({ci[0]:+.3f}..{ci[1]:+.3f}) -> {"HOLDS" if HC4b else "FAILS"}')
say(f'H-C5c genuine cannot-tell <= 0.35: {mixc["cannot tell"]:.3f} -> {"HOLDS" if HC5b else "FAILS"}')
H = {'H-C1c': HC1b, 'H-C2c': HC2b, 'H-C3c': HC3b, 'H-C4c': HC4b, 'H-C5c': HC5b}; say(f'{sum(H.values())} of 5 hold. Failed hypotheses are reported as failed; nothing was retuned. Labels are synthetic; sets describe the score, not proof of manipulation.')
RES = {'protocol_c3_v1_sha256': PHC, 'chosen_transform': CHOSEN, 'hypotheses': H, 'set_mix_B0': mixb, 'set_mix_C3': mixc, 'abnormal_coverage': {'B0': a_b0, 'C3': a_c2}, 'LR': {'B0': LR3b, 'C3': LR3}, 'gain_normal_only': [gain, [float(x) for x in ci]], 'tradeoff_curve': curve, 'mock': MOCK}


--- normal-class coverage on genuine TEST-B bars (B0 = variant R of C1 v3; C3 = chosen score m3) ---
    sym month  bars  B0 normal coverage  C3 normal coverage  target
BTCUSDT   Sep 43200               0.901               0.896     0.9
BTCUSDT   Oct 44640               0.897               0.889     0.9
ETHUSDT   Sep 43200               0.902               0.899     0.9
ETHUSDT   Oct 44640               0.898               0.894     0.9

set mix on genuine TEST-B bars (eps 0.10):
  B0 : {normal} 0.446, cannot tell 0.454, {abnormal} 0.100, {} unlike both 0.000
  C3 : {normal} 0.436, cannot tell 0.459, {abnormal} 0.106, {} unlike both 0.000
abnormal coverage on event bars: B0 0.908, C3 0.909; LR of the abnormal-only set: B0 6.85, C3 6.37

--- trade-off curve (chosen score, normal side eps 0.10; descriptive) ---
 eps abnormal side  abnormal coverage  genuine {normal}  genuine cannot tell  LR abnormal-only
              0.05              0.958             0.249                0.645        

## 7 · Save

In [8]:

(OUT/'c3_results.json').write_text(json.dumps(RES, indent=1, default=str)); (OUT/'c3_share_block.txt').write_text('\n'.join(LINES))
say('Saved: c3_results.json, c3_coverage_table.csv, c3_tradeoff_curve.csv, declared_conformal_v3.json, conformal_tables_v3.npz, c3_share_block.txt in models_b2_v1/ . Send the share block to Claude.')


Saved: c3_results.json, c3_coverage_table.csv, c3_tradeoff_curve.csv, declared_conformal_v3.json, conformal_tables_v3.npz, c3_share_block.txt in models_b2_v1/ . Send the share block to Claude.
